# Tensor hypercontraction

Inspect the three locked THC fixtures, their register bindings and T-equivalent
budgets. Copyright 2023 Google LLC. Adapted from the Qualtran v0.7.0 tutorial
under Apache-2.0.

In [ ]:
from collections import Counter
from io import BytesIO
from math import tau

from IPython.display import Image, SVG, display
from matplotlib import pyplot as plt

from examples.algorithms.chemistry_resource_tutorial import thc_prepare_t_breakdown
from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.thc_chemistry_resources import (
    thc_tutorial_integrals, tutorial_call_model, uniform_t_breakdown, lookup_graph_label,
    uniform_register_score, prepare_register_score, thc_flame_svg,
)
from flagquantum.algorithms.primitives.thc_chemistry_data import thc_chemistry_data
from flagquantum.algorithms.primitives.thc_chemistry_prepare import PrepareTHCChemistry
from flagquantum.algorithms.primitives.thc_chemistry_walk import THCChemistryWalk
from flagquantum.algorithms.primitives.thc_uniform_preparation import THCUniformPreparation
from flagquantum.algorithms.primitives.multiplexers.thc_resource_model import (
    select_thc_cost, select_thc_signature,
)
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.compiler.registers.layout.partition import partition_register
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.register_score_data import register_score_data
from flagquantum.drawer.register_score_mpl import draw_register_score
from flagquantum.dynamic import DynamicCircuit

def show_signature(signature, model):
    display(SVG(field_signature_svg([(model.name, signature, {})])))

def show_calls(model, details=None):
    display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1, details=details))))
    graph = modeled_call_graph(model, max_depth=1)
    for component, count in graph.leaf_counts.items():
        print(component.name, count)

def show_score(score):
    figure, _ = draw_register_score(register_score_data(score), max_width=18, max_height=11)
    with BytesIO() as buffer:
        figure.savefig(buffer, format="png", dpi=120, bbox_inches="tight")
        display(Image(data=buffer.getvalue()))
    plt.close(figure)

def compare_costs(model, paper):
    for family in sorted(model.keys() | paper.keys()):
        print(f"{family:22}  source {model.get(family, 0):4}  paper {paper.get(family, 0):4}")

## Uniform superposition

The original rank-ten example has four spin orbitals. In the source cost
declaration the one-body flag tests the M+1 sentinel. The native circuit
uses its verified alias-domain addresses, with the one-body index at M.

In [ ]:
uniform = THCUniformPreparation(10, 4)
uniform_model = uniform.source_resource_model()
uniform_calls = tutorial_call_model(uniform_model, width=uniform.num_mu.bit_length())

### Signature and call graph

In [ ]:
show_signature(uniform.source_signature, uniform_model)

In [ ]:
show_calls(uniform_calls)

### Folded T-equivalent costs

In [ ]:
display(SVG(thc_flame_svg(uniform_calls)))

In [ ]:
nm = uniform.num_mu.bit_length()
uniform_bins = uniform_t_breakdown(uniform_model)
compare_costs(uniform_bins, {
    "arithmetic": 4 * (4 * (nm - 1) + (4 * nm - 3)),
    "rotations": 4 * (4 + 4),
    "reflection": 4 * (3 + 2 * nm - 1),
    "other": 4 * 3,
})
assert uniform_bins["arithmetic"] == 276

### Source register score

The schedule shows the original predicate, reflection and temporary flag
lifetimes. The drawing describes the frozen composition, including its
sentinel and success logic.

In [ ]:
show_score(uniform_register_score(uniform))

## Signed alias PREPARE

The seed-seven example uses eight THC factors, eight spin orbitals,
eight bits of alias precision, and four rows per lookup block. The
selected and retained lookup cells are separate outputs.

In [ ]:
data = thc_chemistry_data(*thc_tutorial_integrals(), probability_bits=8, log_block_size=2)
prepare = PrepareTHCChemistry(data)
prepare_model = prepare.source_resource_model()
prepare_calls = tutorial_call_model(prepare_model, width=data.probability_bits)
print("Coefficient one-norm", data.normalization)

### Signature and call graph

In [ ]:
show_signature(prepare.source_signature, prepare_model)

In [ ]:
def prepare_labels(component):
    if component.name != "QROAMClean":
        return ""
    widths = tuple(width for _, width in data.target_layout)
    blocks = (1 << data.log_block_size,)
    return lookup_graph_label((data.address_width,), widths, block_sizes=blocks, clean=True)

show_calls(prepare_calls, details=prepare_labels)

### Folded T-equivalent costs

In [ ]:
display(SVG(thc_flame_svg(prepare_calls)))

In [ ]:
prep_bins = thc_prepare_t_breakdown(prepare_model)
n = data.num_mu.bit_length()
compare_costs(prep_bins, {
    "arithmetic": 4 * (n**2 + n - 1) + 4 * data.probability_bits,
    "swaps": 4 * (2 * n + n + 1),
    "data_loading": 4 * ((len(data.source_rows) + 3) // 4
                         + (2 * n + 2 + data.probability_bits) * 3),
})
print("Total source T equivalents", sum(prep_bins.values()))
assert prep_bins["data_loading"] == 248

### Source register score

In [ ]:
show_score(prepare_register_score(prepare))

## SELECT

The original rank-ten, eight-spin-orbital example uses twelve angle bits
and ten alias bits. Its model allocates one angle word and reuses it
for four rotation stages. The frozen rotation leaf declares a budget.

In [ ]:
select_model = select_thc_cost(10, 8, 12, 10)
select_calls = tutorial_call_model(select_model, width=12)

### Signature and call graph

In [ ]:
show_signature(select_thc_signature(10, 8, 10), select_model)

In [ ]:
show_calls(select_calls)

### Folded T-equivalent costs

In [ ]:
display(SVG(thc_flame_svg(select_calls)))

## Bind an executable walk

Supply explicit one- and two-body rotation networks. The executable
reflects about the full prepared signal and preserves its lookup outputs
between steps. The source estimates use a narrower signal and a separate
abstract erasure model.

In [ ]:
import numpy as np

walk = THCChemistryWalk.from_integrals(
    (1.0, -1.0), np.eye(2), np.eye(2),
    networks=(((0, 1, tau / 8),), ((0, 1, -tau / 16),)),
    one_body_networks=((), ((0, 1, tau / 32),)),
    probability_bits=1, log_block_size=0,
)
logical = sum(width for _, width in walk.layout)
work, slots = walk.requirements()
ports = partition_register(tuple(range(logical)), walk.layout)
op = walk.make_on(ports, workspace=tuple(range(logical, logical + work)),
                  classical_bits=tuple(range(slots)))
circuit = DynamicCircuit(logical + work).append(op)
counts = Counter(g.name for g in circuit.to_ir().instructions)
print("Emitted T and T adjoints", counts["t"] + counts["tdg"])
print("Emitted measurements", counts["measure"])
print("Reusable work wires", work)